In [1]:
print("""
=========================================================
CERN Dielectron Dataset

Notebook 6 : Model Training & Evaluation

Objectives
----------
1. Load Engineered Dataset
2. Load Feature Rankings
3. Automatically Generate Feature Subsets
4. Split Dataset
5. Scale Features
6. Prepare Data for Model Training

NOTE:
No models are trained in this part.
This notebook prepares the data pipeline for
systematic model comparison.

Author : Ivy Singh
=========================================================
""")


CERN Dielectron Dataset

Notebook 6 : Model Training & Evaluation

Objectives
----------
1. Load Engineered Dataset
2. Load Feature Rankings
3. Automatically Generate Feature Subsets
4. Split Dataset
5. Scale Features
6. Prepare Data for Model Training

NOTE:
No models are trained in this part.
This notebook prepares the data pipeline for
systematic model comparison.

Author : Ivy Singh



In [2]:
# =====================================================
# Import Required Libraries
# =====================================================

import warnings
warnings.filterwarnings("ignore")

import os
import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# Dataset Split
from sklearn.model_selection import train_test_split

# Feature Scaling
from sklearn.preprocessing import StandardScaler

# Display Settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [3]:
# =====================================================
# Load Engineered Dataset
# =====================================================

df = pd.read_csv("../data/processed/dielectron_engineered.csv")

# Remove accidental spaces
df.columns = df.columns.str.strip()

print("Dataset Loaded Successfully")

print("\nDataset Shape :", df.shape)

Dataset Loaded Successfully

Dataset Shape : (99977, 31)


In [4]:
# =====================================================
# Load Feature Rankings
# =====================================================

feature_rankings = pd.read_csv(
    "../outputs/feature_rankings.csv",
    index_col=0
)

print("Feature Rankings Loaded Successfully")

display(feature_rankings.head())

Feature Rankings Loaded Successfully


,Rank,Correlation,Variance,Mutual Information,Random Forest,Permutation,SHAP,Consensus Score
Eta_Difference,1,0.8444,0.0002,1.0000,0.9345,0.9289,1.0000,0.7847
Total_PT,2,1.0000,0.0790,0.7260,1.0000,1.0000,0.8885,0.7822
Total_Momentum,3,0.7475,0.8721,0.8546,0.1073,0.0779,0.1485,0.4680
Total_Energy,4,0.7475,0.8721,0.8546,0.1033,0.0748,0.1434,0.4659
Phi_Difference,5,0.4033,0.0002,0.6728,0.1232,0.1240,0.3505,0.2790


In [5]:
# =====================================================
# Extract Ranked Feature List
# =====================================================
# Features are already sorted according to the
# Consensus Score generated in Notebook 5.
# =====================================================

ranked_features = feature_rankings.index.tolist()

print("Total Ranked Features :", len(ranked_features))

print("\nTop 10 Features\n")

for feature in ranked_features[:10]:
    print(feature)

Total Ranked Features : 30

Top 10 Features

Eta_Difference
Total_PT
Total_Momentum
Total_Energy
Phi_Difference
Momentum2
E2
Momentum1
E1
pz2


In [6]:
# =====================================================
# Generate Feature Subsets Automatically
# =====================================================
# Different feature subset sizes will be evaluated.
#
# Instead of manually selecting Top-10 or Top-15,
# the workflow remains completely data-driven.
# =====================================================

available_features = len(ranked_features)

candidate_sizes = [5, 8, 10, 12, 15, 18, 20, 25]

feature_sizes = sorted(
    set(
        size
        for size in candidate_sizes
        if size <= available_features
    )
)

# Always include all available features
if available_features not in feature_sizes:
    feature_sizes.append(available_features)

feature_subsets = {
    size: ranked_features[:size]
    for size in feature_sizes
}

print("Feature Subsets Created\n")

for size in feature_sizes:
    print(f"Top {size} Features")

Feature Subsets Created

Top 5 Features
Top 8 Features
Top 10 Features
Top 12 Features
Top 15 Features
Top 18 Features
Top 20 Features
Top 25 Features
Top 30 Features


In [7]:
# =====================================================
# Dataset Overview
# =====================================================

print("="*60)

print("Rows :", df.shape[0])

print("Columns :", df.shape[1])

print("="*60)

print("\nTarget Variable")

print(df["M"].describe())

Rows : 99977
Columns : 31

Target Variable
count   99977.0000
mean       30.0135
std        25.2465
min         2.0001
25%        12.4544
50%        21.2831
75%        38.9927
max       109.9990
Name: M, dtype: float64


In [8]:
# =====================================================
# Train-Test Split Function
# =====================================================
# A reusable function is created so that every
# feature subset is split in exactly the same way.
#
# This guarantees fair model comparison.
# =====================================================

def prepare_dataset(selected_features):

    X = df[selected_features]

    y = df["M"]

    X_train, X_test, y_train, y_test = train_test_split(

        X,

        y,

        test_size=0.20,

        random_state=42

    )

    return X_train, X_test, y_train, y_test

In [9]:
# =====================================================
# Standardization Function
# =====================================================
# IMPORTANT
#
# The scaler is fitted ONLY on the training data.
#
# This prevents data leakage and ensures that
# information from the test set does not influence
# the training process.
# =====================================================

def scale_dataset(X_train, X_test):

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)

    X_test_scaled = scaler.transform(X_test)

    X_train_scaled = pd.DataFrame(

        X_train_scaled,

        columns=X_train.columns,

        index=X_train.index

    )

    X_test_scaled = pd.DataFrame(

        X_test_scaled,

        columns=X_test.columns,

        index=X_test.index

    )

    return scaler, X_train_scaled, X_test_scaled

In [10]:
# =====================================================
# Verify Data Preparation Pipeline
# =====================================================
# Test the pipeline using the smallest feature subset.
# =====================================================

smallest_subset = feature_sizes[0]

features = feature_subsets[smallest_subset]

X_train, X_test, y_train, y_test = prepare_dataset(features)

scaler, X_train_scaled, X_test_scaled = scale_dataset(
    X_train,
    X_test
)

print("="*60)

print("Verification Successful")

print("="*60)

print("Feature Subset :", smallest_subset)

print("Training Shape :", X_train_scaled.shape)

print("Testing Shape :", X_test_scaled.shape)

Verification Successful
Feature Subset : 5
Training Shape : (79981, 5)
Testing Shape : (19996, 5)


In [11]:
# =====================================================
# Save Scaler
# =====================================================
# The scaler fitted on the verification subset is
# saved as an example. During model training, a new
# scaler will be fitted for each feature subset.
# =====================================================

os.makedirs("../models", exist_ok=True)

joblib.dump(
    scaler,
    "../models/standard_scaler.pkl"
)

print("Scaler Saved Successfully")

Scaler Saved Successfully


In [12]:
# =====================================================
# Notebook Summary
# =====================================================

print("="*70)

print("""
Data Preparation Completed Successfully

Completed Steps

✓ Loaded Engineered Dataset

✓ Loaded Feature Rankings

✓ Generated Feature Subsets Automatically

✓ Created Train-Test Split Function

✓ Created Scaling Function

✓ Verified Data Pipeline

Ready for Model Training.
""")

print("="*70)


Data Preparation Completed Successfully

Completed Steps

✓ Loaded Engineered Dataset

✓ Loaded Feature Rankings

✓ Generated Feature Subsets Automatically

✓ Created Train-Test Split Function

✓ Created Scaling Function

✓ Verified Data Pipeline

Ready for Model Training.



In [14]:
# =====================================================
# Import Machine Learning Models
# =====================================================
# We evaluate a diverse set of regression algorithms,
# ranging from simple linear models to advanced
# ensemble methods.
#
# The goal is to identify the model that best predicts
# the invariant mass of dielectron events.
# =====================================================

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

from sklearn.tree import DecisionTreeRegressor

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor
)

from xgboost import XGBRegressor

from lightgbm import LGBMRegressor

from catboost import CatBoostRegressor

print("Models Imported Successfully")

Models Imported Successfully


In [15]:
# =====================================================
# Import Evaluation Metrics
# =====================================================
# These metrics will be used to compare all models.
# =====================================================

from sklearn.metrics import (

    mean_absolute_error,

    mean_squared_error,

    mean_absolute_percentage_error,

    r2_score

)

from sklearn.model_selection import cross_val_score

print("Evaluation Metrics Imported Successfully")

Evaluation Metrics Imported Successfully


In [16]:
# =====================================================
# Define Regression Models
# =====================================================
# All models are stored inside a dictionary.
#
# This allows automatic training using loops,
# avoiding repetitive code.
# =====================================================

models = {

    "Linear Regression": LinearRegression(),

    "Ridge Regression": Ridge(random_state=42),

    "Lasso Regression": Lasso(random_state=42),

    "ElasticNet": ElasticNet(random_state=42),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(

        n_estimators=300,

        random_state=42,

        n_jobs=-1

    ),

    "Extra Trees": ExtraTreesRegressor(

        n_estimators=300,

        random_state=42,

        n_jobs=-1

    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    ),

    "XGBoost": XGBRegressor(

        random_state=42,

        objective="reg:squarederror",

        n_estimators=300,

        verbosity=0

    ),

    "LightGBM": LGBMRegressor(

        random_state=42,

        n_estimators=300,

        verbose=-1

    ),

    "CatBoost": CatBoostRegressor(

        random_state=42,

        verbose=False

    )

}

print("Total Models :", len(models))

Total Models : 11


In [17]:
# =====================================================
# Train and Evaluate Model
# =====================================================
# This function
#
# 1. Trains the model
# 2. Makes predictions
# 3. Computes evaluation metrics
# 4. Returns results
# =====================================================

def evaluate_model(

    model,

    X_train,

    X_test,

    y_train,

    y_test

):

    # Train
    model.fit(
        X_train,
        y_train
    )

    # Prediction
    predictions = model.predict(X_test)

    # Evaluation Metrics
    mae = mean_absolute_error(
        y_test,
        predictions
    )

    mse = mean_squared_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_test,
        predictions
    )

    mape = mean_absolute_percentage_error(
        y_test,
        predictions
    )

    adjusted_r2 = 1 - (

        (1-r2)

        *

        (len(y_test)-1)

        /

        (

            len(y_test)

            -

            X_test.shape[1]

            -

            1

        )

    )

    return {

        "MAE": mae,

        "MSE": mse,

        "RMSE": rmse,

        "R2": r2,

        "Adjusted R2": adjusted_r2,

        "MAPE": mape

    }

In [18]:
# =====================================================
# Train All Models
# =====================================================
# Every model is trained on every feature subset.
#
# Example
#
# Top 5 Features
# ↓
# 11 Models
#
# Top 8 Features
# ↓
# 11 Models
#
# ...
#
# Results are stored inside a dataframe.
# =====================================================

results = []

for subset_size, selected_features in feature_subsets.items():

    print(f"\nTraining using Top {subset_size} Features")

    X_train, X_test, y_train, y_test = prepare_dataset(
        selected_features
    )

    scaler, X_train, X_test = scale_dataset(
        X_train,
        X_test
    )

    for model_name, model in models.items():

        metrics = evaluate_model(

            model,

            X_train,

            X_test,

            y_train,

            y_test

        )

        metrics["Model"] = model_name

        metrics["Features"] = subset_size

        results.append(metrics)

print("\nTraining Completed Successfully")


Training using Top 5 Features

Training using Top 8 Features

Training using Top 10 Features

Training using Top 12 Features

Training using Top 15 Features

Training using Top 18 Features

Training using Top 20 Features

Training using Top 25 Features

Training using Top 30 Features

Training Completed Successfully
